# A3 ABO · 03 Test

Chấm held-out test sau khi nhóm chốt cấu hình dựa trên validation. Giữ cùng thiết bị test khi chạy tiếp một suite đang chấm dở.

Colab dùng A3_ABO_team.zip trong MyDrive/A3_ABO. Jupyter local chạy từ checkout A3 hoặc đặt biến A3_REPO. Dùng cùng A3_WORK, tên cohort và A3_RUN_ID cho các notebook.

In [ ]:
from pathlib import Path
import hashlib
import importlib.util
import json
import os
import shutil
import subprocess
import sys

ON_COLAB = importlib.util.find_spec("google.colab") is not None if importlib.util.find_spec("google") else False
if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

# Edit paths here if using another Drive folder or a checkout outside the current directory.
WORK = Path(os.environ.get("A3_WORK", "/content/drive/MyDrive/A3_ABO" if ON_COLAB else "runs/a3/team")).resolve()
REPO = Path(os.environ.get("A3_REPO", ".")).resolve()
if not (REPO / "src/dlbench/a3/curation.py").is_file():
    if not ON_COLAB and (REPO.parent / "src/dlbench/a3/curation.py").is_file():
        REPO = REPO.parent
    elif ON_COLAB:
        bundle = WORK / "A3_ABO_team.zip"
        if bundle.is_file():
            digest = hashlib.sha256(bundle.read_bytes()).hexdigest()[:16]
            REPO = Path("/content") / f"a3-source-{digest}"
            if not (REPO / "src/dlbench/a3/curation.py").is_file():
                shutil.unpack_archive(bundle, REPO)
        elif (WORK / "setup.py").is_file() and (WORK / "src/dlbench/a3/team_batches.py").is_file():
            REPO = WORK
        else:
            raise FileNotFoundError(f"Đặt A3_ABO_team.zip trong WORK hoặc chỉnh WORK. Đang dùng: {WORK}")
    else:
        raise FileNotFoundError("Set A3_REPO to the A3 checkout or extracted source ZIP.")

if ON_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO}[a3]"], check=True)
source = str(REPO / "src")
if source not in sys.path:
    sys.path.insert(0, source)
env = {**os.environ, "PYTHONPATH": source + os.pathsep + os.environ.get("PYTHONPATH", "")}

def run_a3(*args):
    subprocess.run([sys.executable, "-m", "dlbench.a3.cli", *map(str, args)],
                   cwd=REPO, env=env, check=True)

WORK.mkdir(parents=True, exist_ok=True)
DATA = WORK / "data"
IMAGE_ROOT = DATA / "images"
FROZEN = WORK / "cohorts" / "abo-v1"
IMAGE_CACHE = FROZEN / "image.npz"
TITLE_CACHE = FROZEN / "title.npz"
MANIFEST = FROZEN / "manifest.csv"
RUN_ID = os.environ.get("A3_RUN_ID", "abo-v1-main")
SUITE = WORK / "runs" / RUN_ID
INPUT_ARGS = ("--manifest", MANIFEST, "--image-cache", IMAGE_CACHE, "--title-cache", TITLE_CACHE)
print(f"Source: {REPO}\nWorkspace: {WORK}")

def require_cohort(*paths):
    missing = [path.name for path in paths if not path.is_file()]
    if not missing:
        return
    decisions = WORK / "reviews" / "abo-v1" / "decisions.csv"
    if "freeze.json" in missing or "manifest.csv" in missing:
        if FROZEN.exists():
            action = "Thư mục cohort đang thiếu file. Chép lại nguyên cohort đã chốt hoặc chỉnh WORK."
        elif decisions.is_file():
            action = (f"Audit đã có tại {decisions}. Duyệt và lưu file, rồi điền REVIEWER và "
                      "chạy ô chốt manifest trong Notebook 01 Prepare.")
        else:
            action = "Chạy Notebook 01 Prepare để chốt dữ liệu, hoặc chỉnh WORK tới workspace đã chuẩn bị."
    else:
        action = "Chạy ô tạo cache trong Notebook 01 Prepare với ENCODE = True."
    raise RuntimeError(f"Thiếu {', '.join(missing)} ở {FROZEN}. {action}")

In [ ]:
from dlbench.a3.curation import verified_freeze

TEST_DEVICE = os.environ.get("A3_TEST_DEVICE", "cpu")
require_cohort(FROZEN / "freeze.json", MANIFEST, IMAGE_CACHE, TITLE_CACHE)
verified_freeze(FROZEN)
run_a3("test-team", *INPUT_ARGS, "--output", SUITE, "--device", TEST_DEVICE)
comparison = json.loads((SUITE / "comparison.json").read_text(encoding="utf-8"))
for model, values in comparison["macro_f1"].items():
    print(model, values)
print(SUITE / "comparison.json")